# Q5 — Structured FRN Pricing: Displaced Black Cap and Floor Strips
**SMM269 Fixed Income | UniCredit IT0005599110 | Trade date: 5 Nov 2025**

## Coupon decomposition
$$c(T_i) = \min\!\bigl(\max(1.6\,L_{3M},\;0\%)\,,\;5.45\%\bigr)
= 1.6\,L_{3M}
+ 1.6\,\max(0\% - L_{3M},\,0)
- 1.6\,\max(L_{3M} - 3.4063\%,\,0)$$

The bondholder is **long** an uncapped FRN (1.6 × L), **long** a floor strip (guarantees the coupon cannot go negative), and **short** a cap strip (surrenders upside above 5.45%).

## Vol convention
One flat vol is looked up at the bond's full maturity and used for every caplet in the strip — the standard market convention (Fusai Pricing Caps slide 29; Case Study slide 23).

## Files required (same directory)
| File | Contents |
|---|---|
| `fi_calendar.py` | Calendar and day-count helpers |
| `fi_curve.py` | Curve I/O and shifting |
| `fi_bond.py` | Bond constants, schedule, vol surface, pricer |
| `Interp_term_structure.xlsx` | Bootstrapped daily zero curve (Q3 output) |
| `Holidays.xlsx` | TARGET holiday calendar |
| `Shifted_Black_vol_surface.xlsx` | ICAP VCAP3A surface, shift = 3%, 05-Nov-2025 |


---
## Imports

In [19]:
import os, sys, math
import numpy as np
import pandas as pd
from datetime import date

_here = os.path.dirname(os.path.abspath(__file__)) if "__file__" in dir() else os.getcwd()
sys.path.insert(0, _here)

from fi_calendar import (
    load_target_holidays, modified_following, add_business_days,
    days_30_360, yearfrac_act_360, yearfrac_act_365,
)
from fi_curve import load_curve, make_df_fn, shift_curve
from fi_bond import (
    build_schedule, load_vol_surface, get_flat_vol,
    displaced_black, price_bond, accrued_interest,
    TRADE_DATE, SPOT_LAG, ISSUE_DATE, MATURITY_DATE,
    NOTIONAL, PARTICIPATION, CURRENT_COUPON_RATE, CURRENT_PERIOD_START,
    K_EFF_CAP, K_EFF_FLOOR, SHIFT,
)

HOLIDAYS_FILE       = os.path.join(_here, "Holidays.xlsx")
TERM_STRUCTURE_FILE = os.path.join(_here, "Interp_term_structure.xlsx")
VOL_SURFACE_FILE    = os.path.join(_here, "Shifted_Black_vol_surface.xlsx")
VOL_SURFACE_SHEET   = "6M"

print("Modules loaded: fi_calendar | fi_curve | fi_bond")


Modules loaded: fi_calendar | fi_curve | fi_bond


---
## Load market data

In [20]:
holidays    = load_target_holidays(HOLIDAYS_FILE)
curve       = load_curve(TERM_STRUCTURE_FILE)
vol_surface = load_vol_surface(VOL_SURFACE_FILE, VOL_SURFACE_SHEET)
spot_date   = add_business_days(TRADE_DATE, SPOT_LAG, holidays)
schedule    = build_schedule(ISSUE_DATE, MATURITY_DATE, holidays)

print(f"Spot date        : {spot_date}")
print(f"Curve rows       : {len(curve):,}")
print(f"Vol surface rows : {len(vol_surface)}")
print(f"Coupon periods   : {len(schedule)}")


Spot date        : 2025-11-07
Curve rows       : 18,263
Vol surface rows : 16
Coupon periods   : 40


---
## Vol lookup

In [21]:
# One vol per structure, looked up at the bond's full maturity.
# This single vol is applied uniformly to all caplets / floorlets in the strip.
final_payment  = modified_following(MATURITY_DATE, holidays)
cap_maturity_y = yearfrac_act_365(spot_date, final_payment)

sigma_cap = get_flat_vol(vol_surface, cap_maturity_y, K_EFF_CAP)
sigma_flr = get_flat_vol(vol_surface, cap_maturity_y, K_EFF_FLOOR)

print(f"Cap/floor maturity : {cap_maturity_y:.4f}Y")
print(f"Cap  vol (K={K_EFF_CAP:.4%}) : {sigma_cap:.4%}")
print(f"Floor vol (K={K_EFF_FLOOR:.4%}) : {sigma_flr:.4%}  (higher: left skew at K=0)")
print()
print("Convention: one flat vol per structure, all caplets identical.")
print("Source: Fusai Case Study slide 23; Pricing Caps slide 29.")


Cap/floor maturity : 8.6000Y
Cap  vol (K=3.4062%) : 8.2007%
Floor vol (K=0.0000%) : 11.8511%  (higher: left skew at K=0)

Convention: one flat vol per structure, all caplets identical.
Source: Fusai Case Study slide 23; Pricing Caps slide 29.


---
## Base-case pricing

In [22]:
df_fn_base = make_df_fn(curve, spot_date)

result = price_bond(
    df_fn     = df_fn_base,
    holidays  = holidays,
    schedule  = schedule,
    spot_date = spot_date,
    sigma_cap = sigma_cap,
    sigma_flr = sigma_flr,
)

ai = accrued_interest(TRADE_DATE, CURRENT_PERIOD_START, CURRENT_COUPON_RATE, NOTIONAL)

print("=" * 56)
print(f"  PV(FRN coupons)   :  EUR {result['pv_frn']:>12,.4f}")
print(f"  PV(floor strip)   : +EUR {result['pv_floor']:>12,.4f}")
print(f"  PV(cap strip)     : -EUR {result['pv_cap']:>12,.4f}")
print(f"  PV(notional)      :  EUR {result['pv_notional']:>12,.4f}")
print("  " + "-" * 42)
print(f"  RF gross price    :  EUR {result['gross_price']:>12,.4f}")
print(f"  Accrued interest  : -EUR {ai:>12,.4f}")
print(f"  RF clean price    :  EUR {result['gross_price'] - ai:>12,.4f}")
print("=" * 56)
g = result['gross_price']
print(f"\nComponent breakdown (% of RF gross):")
print(f"  FRN coupons  : {result['pv_frn']   / g:>8.2%}")
print(f"  Floor strip  : {result['pv_floor'] / g:>8.2%}"
      "  (small but non-zero: Displaced Black assigns positive")
print(f"  {'':13s}   probability to negative EURIBOR rates)")
print(f"  Cap strip    : {-result['pv_cap']  / g:>8.2%}")
print(f"  Notional     : {result['pv_notional'] / g:>8.2%}")

# Exported for downstream notebooks
BASE_GROSS_PRICE = result['gross_price']
SIGMA_CAP        = sigma_cap
SIGMA_FLR        = sigma_flr
print(f"\nExported: BASE_GROSS_PRICE = {BASE_GROSS_PRICE:.6f}")


  PV(FRN coupons)   :  EUR     316.8703
  PV(floor strip)   : +EUR       0.3686
  PV(cap strip)     : -EUR      17.3461
  PV(notional)      :  EUR     802.3783
  ------------------------------------------
  RF gross price    :  EUR   1,102.2711
  Accrued interest  : -EUR       4.7794
  RF clean price    :  EUR   1,097.4917

Component breakdown (% of RF gross):
  FRN coupons  :   28.75%
  Floor strip  :    0.03%  (small but non-zero: Displaced Black assigns positive
                  probability to negative EURIBOR rates)
  Cap strip    :   -1.57%
  Notional     :   72.79%

Exported: BASE_GROSS_PRICE = 1102.271139


---
## Diagnostic tables (FRN leg, cap strip, floor strip)

In [23]:
frn_rows, cap_rows, flr_rows = [], [], []

for p in schedule:
    reset_date, start_date = p['Reset Date'], p['Start Date']
    end_date,   pay_date   = p['End Date'],   p['Payment Date']
    if pay_date <= spot_date:
        continue

    df_end       = df_fn_base(pay_date)
    coupon_days  = days_30_360(start_date, end_date)
    coupon_alpha = coupon_days / 360.0

    if reset_date < spot_date:
        fwd_rate    = float('nan')
        coupon_rate = CURRENT_COUPON_RATE
        source      = 'known fixing'
        df_start    = float('nan')
    else:
        df_start    = df_fn_base(start_date)
        fwd_alpha   = yearfrac_act_360(start_date, end_date)
        F           = (df_start / df_end - 1.0) / fwd_alpha
        fwd_rate    = F
        coupon_rate = PARTICIPATION * F
        source      = 'implied forward'

        alpha = fwd_alpha
        T     = yearfrac_act_365(spot_date, start_date)
        F_s   = F + SHIFT

        # d1 / d2 for cap
        K_s_c = K_EFF_CAP + SHIFT
        if T > 0 and F_s > 0 and K_s_c > 0:
            st  = math.sqrt(T)
            d1c = (math.log(F_s / K_s_c) + 0.5 * sigma_cap**2 * T) / (sigma_cap * st)
            d2c = d1c - sigma_cap * st
        else:
            d1c = d2c = float('nan')

        # d1 / d2 for floor
        K_s_f = K_EFF_FLOOR + SHIFT
        if T > 0 and F_s > 0 and K_s_f > 0:
            st  = math.sqrt(T)
            d1f = (math.log(F_s / K_s_f) + 0.5 * sigma_flr**2 * T) / (sigma_flr * st)
            d2f = d1f - sigma_flr * st
        else:
            d1f = d2f = float('nan')

        cup = displaced_black(F, K_EFF_CAP,   T, sigma_cap, df_end, alpha, phi=+1)
        flp = displaced_black(F, K_EFF_FLOOR, T, sigma_flr, df_end, alpha, phi=-1)

        cap_rows.append({
            'Reset Date': reset_date, 'Start Date': start_date, 'Payment Date': pay_date,
            'Fwd Rate': F, 'T reset (y)': T, 'K_eff': K_EFF_CAP, 'Flat vol': sigma_cap,
            'd1': d1c, 'd2': d2c, 'DF payment': df_end,
            'Caplet (unit)': cup, 'Caplet PV': PARTICIPATION * NOTIONAL * cup,
        })
        flr_rows.append({
            'Reset Date': reset_date, 'Start Date': start_date, 'Payment Date': pay_date,
            'Fwd Rate': F, 'T reset (y)': T, 'K_eff': K_EFF_FLOOR, 'Flat vol': sigma_flr,
            'd1': d1f, 'd2': d2f, 'DF payment': df_end,
            'Floorlet (unit)': flp, 'Floorlet PV': PARTICIPATION * NOTIONAL * flp,
        })

    frn_rows.append({
        'Reset Date': reset_date, 'Start Date': start_date, 'Payment Date': pay_date,
        'Days (30/360)': coupon_days, 'Accrual (30/360)': coupon_alpha,
        'DF start': df_start, 'DF end': df_end,
        'Fwd Rate (ACT/360)': fwd_rate, 'Coupon rate (x1.6)': coupon_rate,
        'Cash flow': NOTIONAL * coupon_rate * coupon_alpha,
        'PV of cash flow': NOTIONAL * coupon_rate * coupon_alpha * df_end,
        '_source': source,
    })

frn_df = pd.DataFrame(frn_rows)
cap_df = pd.DataFrame(cap_rows)
flr_df = pd.DataFrame(flr_rows)

print(f"FRN leg:   {len(frn_df)} coupons   PV = EUR {frn_df['PV of cash flow'].sum():,.4f}")
print(f"Cap strip: {len(cap_df)} caplets   PV = EUR {cap_df['Caplet PV'].sum():,.4f}")
print(f"Flr strip: {len(flr_df)} floorlets PV = EUR {flr_df['Floorlet PV'].sum():,.4f}")


FRN leg:   35 coupons   PV = EUR 316.8703
Cap strip: 34 caplets   PV = EUR 17.3461
Flr strip: 34 floorlets PV = EUR 0.3686


### FRN coupon leg

In [24]:
display(frn_df.drop(columns=['_source']).style.format({
    'Accrual (30/360)':    '{:.6f}',
    'DF start':            '{:.6f}',
    'DF end':              '{:.6f}',
    'Fwd Rate (ACT/360)':  '{:.6%}',
    'Coupon rate (x1.6)':  '{:.6%}',
    'Cash flow':           '{:,.4f}',
    'PV of cash flow':     '{:,.4f}',
}))


,Reset Date,Start Date,Payment Date,Days (30/360),Accrual (30/360),DF start,DF end,Fwd Rate (ACT/360),Coupon rate (x1.6),Cash flow,PV of cash flow
0,2025-09-10,2025-09-12,2025-12-12,90,0.250000,nan,0.998136,nan%,3.246400%,8.1160,8.1009
1,2025-12-10,2025-12-12,2026-03-12,90,0.250000,0.998136,0.992880,2.117452%,3.387924%,8.4698,8.4095
2,2026-03-10,2026-03-12,2026-06-12,90,0.250000,0.992880,0.987299,2.211864%,3.538983%,8.8475,8.7351
3,2026-06-10,2026-06-12,2026-09-14,92,0.255556,0.987299,0.982070,2.039413%,3.263061%,8.3389,8.1894
4,2026-09-10,2026-09-14,2026-12-14,90,0.250000,0.982070,0.977068,2.025087%,3.240138%,8.1003,7.9146
5,2026-12-10,2026-12-14,2027-03-12,88,0.244444,0.977068,0.972161,2.064882%,3.303812%,8.0760,7.8512
6,2027-03-10,2027-03-12,2027-06-14,92,0.255556,0.972161,0.966883,2.090569%,3.344911%,8.5481,8.2650
7,2027-06-10,2027-06-14,2027-09-13,89,0.247222,0.966883,0.961661,2.148241%,3.437186%,8.4975,8.1717
8,2027-09-09,2027-09-13,2027-12-13,90,0.250000,0.961661,0.956400,2.176319%,3.482111%,8.7053,8.3257
9,2027-12-09,2027-12-13,2028-03-13,90,0.250000,0.956400,0.951075,2.214754%,3.543606%,8.8590,8.4256


### Cap strip (short: reduces bond value)

In [25]:
display(cap_df.style.format({
    'Fwd Rate':       '{:.6%}', 'T reset (y)': '{:.4f}',
    'K_eff':          '{:.6%}', 'Flat vol':    '{:.4%}',
    'd1':             '{:.4f}', 'd2':          '{:.4f}',
    'DF payment':     '{:.6f}', 'Caplet (unit)': '{:.8f}',
    'Caplet PV':      '{:,.4f}',
}).set_caption(
    f'Cap strip: K_eff = {K_EFF_CAP:.4%}, '
    f'flat vol = {sigma_cap:.4%}  (at {cap_maturity_y:.2f}Y maturity)'
))


,Reset Date,Start Date,Payment Date,Fwd Rate,T reset (y),K_eff,Flat vol,d1,d2,DF payment,Caplet (unit),Caplet PV
0,2025-12-10,2025-12-12,2026-03-12,2.117452%,0.0959,3.406250%,8.2007%,-8.8325,-8.8579,0.992880,0.00000000,0.0000
1,2026-03-10,2026-03-12,2026-06-12,2.211864%,0.3425,3.406250%,8.2007%,-4.2755,-4.3235,0.987299,0.00000000,0.0000
2,2026-06-10,2026-06-12,2026-09-14,2.039413%,0.5945,3.406250%,8.2007%,-3.7637,-3.8270,0.982070,0.00000002,0.0000
3,2026-09-10,2026-09-14,2026-12-14,2.025087%,0.8521,3.406250%,8.2007%,-3.1701,-3.2458,0.977068,0.00000019,0.0003
4,2026-12-10,2026-12-14,2027-03-12,2.064882%,1.1014,3.406250%,8.2007%,-2.6869,-2.7729,0.972161,0.00000112,0.0018
5,2027-03-10,2027-03-12,2027-06-14,2.090569%,1.3425,3.406250%,8.2007%,-2.3719,-2.4669,0.966883,0.00000351,0.0056
6,2027-06-10,2027-06-14,2027-09-13,2.148241%,1.6000,3.406250%,8.2007%,-2.0557,-2.1594,0.961661,0.00000916,0.0147
7,2027-09-09,2027-09-13,2027-12-13,2.176319%,1.8493,3.406250%,8.2007%,-1.8558,-1.9673,0.956400,0.00001663,0.0266
8,2027-12-09,2027-12-13,2028-03-13,2.214754%,2.0986,3.406250%,8.2007%,-1.6728,-1.7916,0.951075,0.00002786,0.0446
9,2028-03-09,2028-03-13,2028-06-12,2.238985%,2.3479,3.406250%,8.2007%,-1.5379,-1.6636,0.945723,0.00004032,0.0645


### Floor strip (long: increases bond value)

In [26]:
display(flr_df.style.format({
    'Fwd Rate':         '{:.6%}', 'T reset (y)': '{:.4f}',
    'K_eff':            '{:.6%}', 'Flat vol':    '{:.4%}',
    'd1':               '{:.4f}', 'd2':          '{:.4f}',
    'DF payment':       '{:.6f}', 'Floorlet (unit)': '{:.8f}',
    'Floorlet PV':      '{:,.4f}',
}).set_caption(
    f'Floor strip: K_eff = {K_EFF_FLOOR:.4%}, '
    f'flat vol = {sigma_flr:.4%}  (left-skew: higher vol than cap)'
))


,Reset Date,Start Date,Payment Date,Fwd Rate,T reset (y),K_eff,Flat vol,d1,d2,DF payment,Floorlet (unit),Floorlet PV
0,2025-12-10,2025-12-12,2026-03-12,2.117452%,0.0959,0.000000%,11.8511%,14.5706,14.5339,0.992880,0.00000000,0.0000
1,2026-03-10,2026-03-12,2026-06-12,2.211864%,0.3425,0.000000%,11.8511%,7.9986,7.9293,0.987299,0.00000000,0.0000
2,2026-06-10,2026-06-12,2026-09-14,2.039413%,0.5945,0.000000%,11.8511%,5.7219,5.6305,0.982070,0.00000000,0.0000
3,2026-09-10,2026-09-14,2026-12-14,2.025087%,0.8521,0.000000%,11.8511%,4.7700,4.6607,0.977068,0.00000000,0.0000
4,2026-12-10,2026-12-14,2027-03-12,2.064882%,1.1014,0.000000%,11.8511%,4.2731,4.1487,0.972161,0.00000000,0.0000
5,2027-03-10,2027-03-12,2027-06-14,2.090569%,1.3425,0.000000%,11.8511%,3.9196,3.7822,0.966883,0.00000002,0.0000
6,2027-06-10,2027-06-14,2027-09-13,2.148241%,1.6000,0.000000%,11.8511%,3.6775,3.5276,0.961661,0.00000006,0.0001
7,2027-09-09,2027-09-13,2027-12-13,2.176319%,1.8493,0.000000%,11.8511%,3.4652,3.3041,0.956400,0.00000014,0.0002
8,2027-12-09,2027-12-13,2028-03-13,2.214754%,2.0986,0.000000%,11.8511%,3.3062,3.1345,0.951075,0.00000028,0.0004
9,2028-03-09,2028-03-13,2028-06-12,2.238985%,2.3479,0.000000%,11.8511%,3.1609,2.9793,0.945723,0.00000051,0.0008
